# Fine-tune ArabicOCR CRNN-CTC trên KHATT

Notebook này fine-tune checkpoint `FixFips/arabicocr-khatt` trên dataset `data_preprocessed` của dự án. Pipeline giữ nguyên kiến trúc CRNN-CTC v2, preprocessing lúc train/inference, chiều RTL và định dạng checkpoint của package.

## Trước khi chạy trên Kaggle

1. Upload `data_preprocessed.zip` thành một **private Kaggle Dataset** rồi **Add Input** vào notebook; chỉ public nếu giấy phép KHATT của bạn cho phép phân phối lại.
2. Chọn accelerator GPU (T4/P100 hoặc tốt hơn).
3. Bật Internet để cài package và tải checkpoint pretrained từ Hugging Face.
4. Chọn **Run All**. Model tốt nhất và ZIP export nằm trong `/kaggle/working/arabicocr-khatt-finetuned/`.

> CSV gốc chứa token Latin (`ke`, `la`, `sp`, ...), không phải Arabic Unicode. Notebook giải mã đủ 67 token trước khi huấn luyện.

In [ ]:
%pip install -q arabicocr-khatt==0.1.0 rapidfuzz huggingface-hub

## 1. Cấu hình
Các giá trị mặc định ưu tiên chạy ổn định trên Kaggle T4. Tăng `BATCH_SIZE` nếu GPU còn nhiều VRAM; tăng `EPOCHS` khi validation CER vẫn tiếp tục giảm.

In [ ]:
import csv
import math
import os
import random
import shutil
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from huggingface_hub import hf_hub_download
from rapidfuzz.distance import Levenshtein
from torch.utils.data import DataLoader, Dataset

from arabicocr_khatt.augment import ArabicAugment
from arabicocr_khatt.metrics import norm_ar
from arabicocr_khatt.model import CRNN, build_bigram_lm, ctc_greedy_decode, text_to_ids
from arabicocr_khatt.pipeline import ArabicOCR, load_checkpoint, save_bigram_lm_json
from arabicocr_khatt.preprocess import (
    binarize, normalize, pad_width, resize_keep_ratio_height, to_grayscale,
)

SEED = 42
HEIGHT = 96
MAX_WIDTH = 1536
BATCH_SIZE = 8
GRAD_ACCUM = 4          # effective batch = 32
EPOCHS = 25
PATIENCE = 6
MAX_LR = 3e-4           # fine-tuning LR, thấp hơn train-from-scratch
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
OUTPUT_DIR = Path('/kaggle/working/arabicocr-khatt-finetuned')
PRETRAINED_REPO = 'FixFips/arabicocr-khatt'
PRETRAINED_FILE = 'crnn_best.pt'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE, torch.cuda.get_device_name(0) if AMP_ENABLED else '')

## 2. Tự tìm dataset trong Kaggle Input
Cell này hỗ trợ cả dataset đã được Kaggle giải nén và trường hợp input vẫn là `data_preprocessed.zip`.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
EXTRACT_ROOT = Path('/kaggle/working/khatt-input')
DATA_ROOT_OVERRIDE = None  # ví dụ: Path('/kaggle/input/my-dataset/data_preprocessed')

def is_dataset_root(path: Path) -> bool:
    return (path / 'Train.csv').is_file() and (path / 'Validation.csv').is_file() \
        and (path / 'Train').is_dir() and (path / 'Validation').is_dir()

def find_dataset_root() -> Path:
    if DATA_ROOT_OVERRIDE is not None:
        root = Path(DATA_ROOT_OVERRIDE)
        if not is_dataset_root(root):
            raise FileNotFoundError(f'Invalid DATA_ROOT_OVERRIDE: {root}')
        return root

    for train_csv in INPUT_ROOT.rglob('Train.csv'):
        if is_dataset_root(train_csv.parent):
            return train_csv.parent

    zip_candidates = sorted(INPUT_ROOT.rglob('data_preprocessed.zip'))
    if not zip_candidates:
        zip_candidates = sorted(INPUT_ROOT.rglob('*.zip'))
    for archive in zip_candidates:
        print('Extracting:', archive)
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as bundle:
            bundle.extractall(EXTRACT_ROOT)
        for train_csv in EXTRACT_ROOT.rglob('Train.csv'):
            if is_dataset_root(train_csv.parent):
                return train_csv.parent
    raise FileNotFoundError('Không tìm thấy data_preprocessed. Hãy Add Input trên Kaggle.')

DATA_ROOT = find_dataset_root()
print('Dataset root:', DATA_ROOT)
print('Train images:', len(list((DATA_ROOT / 'Train').glob('*.jpg'))))
print('Validation images:', len(list((DATA_ROOT / 'Validation').glob('*.jpg'))))

## 3. Giải mã nhãn KHATT token → Arabic Unicode
Mapping cơ bản theo bảng Arabic–Latin của KHATT v1.0; bốn mã ligature lam–alef được bổ sung theo dữ liệu. Notebook dừng ngay nếu gặp token chưa biết thay vì âm thầm tạo nhãn sai.

In [ ]:
CODE_MAP = {
    'hh':'ء', 'am':'آ', 'ae':'أ', 'ah':'إ', 'aa':'ا',
    'ba':'ب', 'ta':'ت', 'teE':'ة', 'th':'ث', 'ja':'ج',
    'ha':'ح', 'kh':'خ', 'da':'د', 'dh':'ذ', 'ra':'ر',
    'za':'ز', 'se':'س', 'sh':'ش', 'sa':'ص', 'de':'ض',
    'to':'ط', 'zha':'ظ', 'ay':'ع', 'gh':'غ', 'fa':'ف',
    'ka':'ق', 'ke':'ك', 'la':'ل', 'ma':'م', 'na':'ن',
    'he':'ه', 'wa':'و', 'wl':'ؤ', 'ya':'ي', 'ee':'ى', 'al':'ئ',
    'n0':'0', 'n1':'1', 'n2':'2', 'n3':'3', 'n4':'4',
    'n5':'5', 'n6':'6', 'n7':'7', 'n8':'8', 'n9':'9',
    'atr':'@', 'col':':', 'dbq':'"', 'com':',', 'qts':'?',
    'exc':'!', 'dot':'.', 'bro':'(', 'brc':')', 'fsl':'/',
    'bsl':'\\', 'equ':'=', 'hyp':'-', 'usc':'_', 'scr':'#',
    'per':'%', 'sp':' ',
    'laaa':'لا', 'laam':'لآ', 'laah':'لإ', 'laae':'لأ',
}
CODE_MAP = {key.lower(): value for key, value in CODE_MAP.items()}
SKIP_TOKENS = {';'}

def decode_codes(codes):
    chars, unknown = [], []
    for raw in codes:
        code = raw.strip()
        if not code or code in SKIP_TOKENS:
            continue
        char = CODE_MAP.get(code.lower())
        if char is None:
            unknown.append(code)
        else:
            chars.append(char)
    return ''.join(chars), unknown

def read_split(csv_path: Path, image_dir: Path) -> pd.DataFrame:
    rows, unknown_counts = [], {}
    with csv_path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.reader(stream)
        next(reader, None)
        for row in reader:
            if not row or not row[0].strip():
                continue
            filename = row[0].strip()
            text, unknown = decode_codes(row[1:132])
            for token in unknown:
                unknown_counts[token] = unknown_counts.get(token, 0) + 1
            rows.append({'filename': filename, 'image_path': image_dir / filename, 'text': text})
    if unknown_counts:
        raise ValueError(f'Unknown KHATT tokens: {unknown_counts}')
    frame = pd.DataFrame(rows)
    missing = [str(path) for path in frame.image_path if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(f'{len(missing)} images missing; first: {missing[:5]}')
    if (frame.text.str.len() == 0).any():
        raise ValueError('Dataset contains empty decoded labels')
    return frame

train_df = read_split(DATA_ROOT / 'Train.csv', DATA_ROOT / 'Train')
val_df = read_split(DATA_ROOT / 'Validation.csv', DATA_ROOT / 'Validation')
print(f'Train={len(train_df):,}, Validation={len(val_df):,}')
print('Decoded samples:')
for text in train_df.text.head(5):
    print('  ', text)

## 4. Dataset và CTC collate
Ảnh đã đi qua preprocessing của dự án, nhưng vẫn được binarize/resize/pad bằng chính preprocessing của `arabicocr-khatt` để train và inference khớp nhau. Augmentation chỉ áp dụng cho train và tránh erosion/dilation vì chúng dễ phá hủy dấu chấm Arabic.

In [ ]:
class KHATTFineTuneDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, augment=None):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as opened:
            image = to_grayscale(opened)
            image = binarize(image)
            image = normalize(image)
        if self.augment is not None:
            image = self.augment(image)
        image = resize_keep_ratio_height(image, HEIGHT)
        if image.width > MAX_WIDTH:
            image = image.resize((MAX_WIDTH, HEIGHT), Image.Resampling.LANCZOS)
        image = pad_width(image, MAX_WIDTH)
        array = np.asarray(image, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(array).unsqueeze(0)
        return tensor, row.text, row.filename

class CTCCollate:
    def __init__(self, char2id, unk_id):
        self.char2id = char2id
        self.unk_id = unk_id

    def __call__(self, batch):
        images, texts_rtl, filenames = zip(*batch)
        texts_ltr = [text[::-1] for text in texts_rtl]
        encoded = [
            torch.tensor(text_to_ids(text, self.char2id, self.unk_id), dtype=torch.long)
            for text in texts_ltr
        ]
        lengths = torch.tensor([len(item) for item in encoded], dtype=torch.long)
        targets = torch.cat(encoded)
        return torch.stack(images), targets, lengths, list(texts_rtl), list(filenames)

## 5. Tải checkpoint và mở rộng vocabulary nếu cần
Phần lớn ký tự đã có trong checkpoint. Nếu dataset chứa ký hiệu hiếm như `\`, `=` hoặc `_`, notebook thêm class mới và giữ nguyên toàn bộ trọng số pretrained của các class cũ.

In [ ]:
PRETRAINED_CKPT = Path(hf_hub_download(PRETRAINED_REPO, PRETRAINED_FILE))
base_state = load_checkpoint(PRETRAINED_CKPT)
base_vocab = list(base_state['vocab'])
dataset_chars = sorted(set(''.join(train_df.text) + ''.join(val_df.text)))
missing_chars = [char for char in dataset_chars if char not in base_vocab]
vocab = base_vocab + missing_chars
char2id = {char: index for index, char in enumerate(vocab)}
id2char = {index: char for index, char in enumerate(vocab)}
unk_id = char2id.get('<unk>', 1)
print(f'Base vocab={len(base_vocab)}, expanded vocab={len(vocab)}')
print('New characters:', missing_chars or 'none')

model = CRNN(num_classes=len(vocab), use_attention=False)
fresh_state = model.state_dict()
for key, value in base_state['model'].items():
    if key in fresh_state and fresh_state[key].shape == value.shape:
        fresh_state[key] = value
model.load_state_dict(fresh_state)
if missing_chars:
    with torch.no_grad():
        model.fc.weight[:len(base_vocab)].copy_(base_state['model']['fc.weight'])
        model.fc.bias[:len(base_vocab)].copy_(base_state['model']['fc.bias'])
model = model.to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
collate = CTCCollate(char2id, unk_id)
train_ds = KHATTFineTuneDataset(train_df, augment=ArabicAugment(training=True))
val_ds = KHATTFineTuneDataset(val_df)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED, persistent_workers=NUM_WORKERS > 0, collate_fn=collate,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED, persistent_workers=NUM_WORKERS > 0, collate_fn=collate,
)
print('Train batches:', len(train_loader), 'Validation batches:', len(val_loader))

## 6. Evaluation helpers
CER/WER dưới đây là corpus-level micro average. `WER(norm)` bỏ tashkeel/tatweel và gộp một số biến thể alef để bổ sung góc nhìn, nhưng checkpoint vẫn được chọn theo strict CER.

In [ ]:
def corpus_cer(refs, hyps):
    errors = sum(Levenshtein.distance(list(r), list(h)) for r, h in zip(refs, hyps))
    return errors / max(1, sum(len(r) for r in refs))

def corpus_wer(refs, hyps):
    errors = sum(Levenshtein.distance(r.split(), h.split()) for r, h in zip(refs, hyps))
    return errors / max(1, sum(len(r.split()) for r in refs))

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    refs, hyps, filenames = [], [], []
    for images, _, _, texts_rtl, names in loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        decoded_ltr = ctc_greedy_decode(logits, id2char)
        decoded_rtl = [text[::-1] for text in decoded_ltr]
        refs.extend(texts_rtl)
        hyps.extend(decoded_rtl)
        filenames.extend(names)
    refs_norm = [norm_ar(text) for text in refs]
    hyps_norm = [norm_ar(text) for text in hyps]
    metrics = {
        'cer': corpus_cer(refs, hyps),
        'wer': corpus_wer(refs, hyps),
        'wer_norm': corpus_wer(refs_norm, hyps_norm),
    }
    predictions = pd.DataFrame({'filename': filenames, 'reference': refs, 'prediction': hyps})
    return metrics, predictions

## 7. Fine-tune
Best checkpoint được lưu theo validation strict CER. Early stopping giúp tránh overfit; gradient clipping và CTC `zero_infinity=True` giúp batch lỗi không làm hỏng cả run.

In [ ]:
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR / 10, weight_decay=WEIGHT_DECAY)
updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, epochs=EPOCHS, steps_per_epoch=updates_per_epoch,
    pct_start=0.1, anneal_strategy='cos', div_factor=10, final_div_factor=100,
)
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
BEST_CKPT = OUTPUT_DIR / 'crnn_best.pt'
history = []
best_cer = float('inf')
stale_epochs = 0

for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0

    for step, (images, targets, target_lengths, _, _) in enumerate(train_loader, start=1):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        target_lengths = target_lengths.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
            logits = model(images)
            timesteps, batch, _ = logits.shape
            input_lengths = torch.full((batch,), timesteps, dtype=torch.long, device=DEVICE)
            loss = criterion(logits.float().log_softmax(2), targets, input_lengths, target_lengths)
            scaled_loss = loss / GRAD_ACCUM
        scaler.scale(scaled_loss).backward()
        running_loss += float(loss.detach())

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    val_metrics, val_predictions = evaluate(model, val_loader)
    train_loss = running_loss / max(1, len(train_loader))
    elapsed = time.perf_counter() - started
    row = {
        'epoch': epoch, 'train_loss': train_loss, **val_metrics,
        'lr': optimizer.param_groups[0]['lr'], 'seconds': elapsed,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
    print(
        f"Epoch {epoch:02d} | loss={train_loss:.4f} | CER={val_metrics['cer']:.4f} | "
        f"WER={val_metrics['wer']:.4f} | WER(norm)={val_metrics['wer_norm']:.4f} | {elapsed:.0f}s"
    )

    if val_metrics['cer'] < best_cer:
        best_cer = val_metrics['cer']
        stale_epochs = 0
        torch.save({
            'model': model.state_dict(), 'vocab': vocab,
            'arch_version': 2, 'use_attention': False,
        }, BEST_CKPT)
        val_predictions.to_csv(OUTPUT_DIR / 'validation_predictions.csv', index=False)
        print(f'  Saved best checkpoint: CER={best_cer:.4f}')
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f'Early stopping: {PATIENCE} epochs without CER improvement')
            break

print('Best validation CER:', best_cer)

## 8. Đánh giá best checkpoint và xem learning curve

In [ ]:
best_state = load_checkpoint(BEST_CKPT)
model.load_state_dict(best_state['model'])
best_metrics, best_predictions = evaluate(model, val_loader)
print(best_metrics)
display(best_predictions.sample(min(10, len(best_predictions)), random_state=SEED))

metrics_df = pd.read_csv(OUTPUT_DIR / 'metrics.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(metrics_df.epoch, metrics_df.train_loss, marker='o')
axes[0].set(title='Training loss', xlabel='Epoch', ylabel='CTC loss')
axes[1].plot(metrics_df.epoch, metrics_df.cer, marker='o', label='CER')
axes[1].plot(metrics_df.epoch, metrics_df.wer, marker='o', label='WER')
axes[1].set(title='Validation error', xlabel='Epoch', ylabel='Error rate')
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Build bigram LM và export
Bigram được xây trên chuỗi đã đảo sang chiều thời gian LTR của CTC. Checkpoint có thể dùng trực tiếp với `ArabicOCR(checkpoint=...)`.

In [ ]:
bigram_lm = build_bigram_lm([text[::-1] for text in train_df.text], char2id)
save_bigram_lm_json(bigram_lm, OUTPUT_DIR / 'bigram_lm.json')

ocr = ArabicOCR(BEST_CKPT, device=str(DEVICE), bigram_lm=bigram_lm)
sample = val_df.iloc[0]
prediction = ocr.recognize(sample.image_path, segment=False, beam_width=10, lm_weight=0.3)
print('Reference :', sample.text)
print('Prediction:', prediction)

archive_path = shutil.make_archive(
    '/kaggle/working/arabicocr-khatt-finetuned', 'zip',
    root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name,
)
print('Export directory:', OUTPUT_DIR)
print('Download ZIP:', archive_path)

## 10. Dùng checkpoint sau khi tải về

```python
from arabicocr_khatt import ArabicOCR
from arabicocr_khatt.pipeline import load_bigram_lm_json

lm = load_bigram_lm_json('bigram_lm.json')
ocr = ArabicOCR('crnn_best.pt', device='cuda', bigram_lm=lm)
print(ocr.recognize('line.jpg', segment=False))
```

Nếu ảnh đầu vào là cả trang, bỏ `segment=False` để package tự tách dòng.